### IMPORT LIBRARIES

In [ ]:
import json
import os
import random
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict, Counter
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
from sklearn.metrics import classification_report, confusion_matrix

### LOAD DATA

In [ ]:
# ANNOTATED_FOLDER = "/kaggle/input/annotated/Annotated"  # Folder with prev/next format
ANNOTATED_MINUS_1_FOLDER = "/kaggle/input/annotated-with-negative-1/Annotated_Processed"  # Folder with prev-1/prev format

OUTPUT_DIR = "/kaggle/working"
# COMBINED_JSONL = f"{OUTPUT_DIR}/combined_clauses.jsonl"  # prev/current/next format
COMBINED_PREV_JSONL = f"{OUTPUT_DIR}/combined_clauses_prev.jsonl"  # prev-1/prev/current format

# Split configuration
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
random.seed(SEED)
np.random.seed(SEED)


# LOAD AND COMBINE DATA FROM ANNOTATED FOLDER (prev/current/next format)
def load_annotated_folder(folder_path, use_prev_minus_one=False):
    """
    Load all JSON files from folder and convert to JSONL format.

    Args:
        folder_path: Path to folder containing JSON files
        use_prev_minus_one: If True, use prev_minus_1_clause; else use next_clause

    Returns:
        List of dictionaries, one per clause
    """
    all_clauses = []
    count_docs = 0
    count_clauses = 0

    # Walk through all subdirectories
    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            if filename.endswith(".json"):
                filepath = os.path.join(root, filename)
                try:
                    with open(filepath, "r", encoding="utf-8") as f:
                        data = json.load(f)

                    if "doc_id" not in data or "clauses" not in data:
                        continue

                    doc_id = data["doc_id"]

                    for clause in data["clauses"]:
                        if "text" not in clause or "label" not in clause:
                            continue

                        entry = {
                            "doc_id": doc_id,
                            "clause_id": clause.get("clause_id", 0),
                            "text": clause.get("text", ""),
                            "label": clause.get("label", "None"),
                        }

                        if use_prev_minus_one:
                            # For Annotated_minus_1 folder
                            entry["prev_text"] = clause.get("prev_clause")
                            entry["prev_text_1"] = clause.get("prev_minus_1_clause")
                        else:
                            # For Annotated folder
                            entry["prev_text"] = clause.get("prev_clause")
                            entry["next_text"] = clause.get("next_clause")

                        all_clauses.append(entry)
                        count_clauses += 1

                    count_docs += 1

                except json.JSONDecodeError as e:
                    print(f"⚠️ Error decoding {filepath}: {e}")
                    continue
                except Exception as e:
                    print(f"⚠️ Error processing {filepath}: {e}")
                    continue

    print(f"Loaded {count_docs} documents, {count_clauses} clauses from {folder_path}")
    return all_clauses

# Load from both folders
# print("Loading data from Annotated folder (prev/current/next format)...")
# clauses_annotated = load_annotated_folder(ANNOTATED_FOLDER, use_prev_minus_one=False)

print("\nLoading data from Annotated_minus_1 folder (prev-1/prev/current format)...")
clauses_prev = load_annotated_folder(ANNOTATED_MINUS_1_FOLDER, use_prev_minus_one=True)

print(f"\nTotal clauses loaded:")
# print(f"  From Annotated: {len(clauses_annotated)}")
print(f"  From Annotated_minus_1: {len(clauses_prev)}")


# CHECK LABEL DISTRIBUTION
def print_label_distribution(clauses, name):
    """Print label distribution for a set of clauses."""
    label_counts = Counter(clause["label"] for clause in clauses)
    total = len(clauses)

    print(f"\n{name} Label Distribution:")
    print(f"Total: {total} clauses")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count} ({100*count/total:.2f}%)")

# print_label_distribution(clauses_annotated, "Annotated folder")
print_label_distribution(clauses_prev, "Annotated_minus_1 folder")

# SAVE COMBINED JSONL FILES
def save_jsonl(filepath, clauses):
    """Save clauses to JSONL file."""
    with open(filepath, "w", encoding="utf-8") as f:
        for clause in clauses:
            json.dump(clause, f, ensure_ascii=False)
            f.write("\n")
    print(f"✓ Saved {len(clauses)} clauses to {filepath}")

print("Saving combined JSONL files...")
# save_jsonl(COMBINED_JSONL, clauses_annotated)
save_jsonl(COMBINED_PREV_JSONL, clauses_prev)

print(f"\n✓ Combined files saved:")
# print(f"  {COMBINED_JSONL} (prev/current/next format)")
print(f"  {COMBINED_PREV_JSONL} (prev-1/prev/current format)")

### SPLIT INTO TRAIN/VAL/TEST (Document-level split to avoid data leakage)

In [ ]:
def split_by_documents(clauses, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=42):
    """
    Split clauses by document to avoid data leakage.
    Ensures clauses from the same document stay together.
    """
    random.seed(seed)

    # Group clauses by document
    docs_to_clauses = defaultdict(list)
    for clause in clauses:
        doc_id = clause["doc_id"]
        docs_to_clauses[doc_id].append(clause)

    # Get list of document IDs
    doc_ids = list(docs_to_clauses.keys())
    random.shuffle(doc_ids)

    total_docs = len(doc_ids)
    train_end = int(total_docs * train_ratio)
    val_end = train_end + int(total_docs * val_ratio)

    # Split documents
    train_docs = doc_ids[:train_end]
    val_docs = doc_ids[train_end:val_end]
    test_docs = doc_ids[val_end:]

    train_clauses = []
    val_clauses = []
    test_clauses = []

    for doc_id in train_docs:
        train_clauses.extend(docs_to_clauses[doc_id])

    for doc_id in val_docs:
        val_clauses.extend(docs_to_clauses[doc_id])

    for doc_id in test_docs:
        test_clauses.extend(docs_to_clauses[doc_id])

    # Shuffle clauses within each split
    random.shuffle(train_clauses)
    random.shuffle(val_clauses)
    random.shuffle(test_clauses)

    return train_clauses, val_clauses, test_clauses, train_docs, val_docs, test_docs

# Split the data
print("Splitting data into train/val/test (document-level split)...")
train_clauses, val_clauses, test_clauses, train_docs, val_docs, test_docs = split_by_documents(
    clauses_prev,
    # clauses_annotated,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED
)

print(f"\nSplit Summary:")
print(f"  Documents - Train: {len(train_docs)}, Val: {len(val_docs)}, Test: {len(test_docs)}")
print(f"  Clauses - Train: {len(train_clauses)}, Val: {len(val_clauses)}, Test: {len(test_clauses)}")
print(f"  Total: {len(train_clauses) + len(val_clauses) + len(test_clauses)} clauses")

### CREATE CORRESPONDING SPLITS FOR PREV_MINUS_1 FORMAT

In [ ]:
# Create a mapping from (doc_id, clause_id) to clause for prev format
prev_clause_map = {}
for clause in clauses_prev:
    key = (clause["doc_id"], clause["clause_id"])
    prev_clause_map[key] = clause

train_clauses_prev = []
val_clauses_prev = []
test_clauses_prev = []

for clause in train_clauses:
    key = (clause["doc_id"], clause["clause_id"])
    if key in prev_clause_map:
        train_clauses_prev.append(prev_clause_map[key])

for clause in val_clauses:
    key = (clause["doc_id"], clause["clause_id"])
    if key in prev_clause_map:
        val_clauses_prev.append(prev_clause_map[key])

for clause in test_clauses:
    key = (clause["doc_id"], clause["clause_id"])
    if key in prev_clause_map:
        test_clauses_prev.append(prev_clause_map[key])

print(f"Created corresponding splits for prev_minus_1 format:")
print(f"  Train: {len(train_clauses_prev)}, Val: {len(val_clauses_prev)}, Test: {len(test_clauses_prev)}")

# SAVE TRAIN/VAL/TEST SPLITS
# # Save splits for prev/current/next format
# save_jsonl(f"{OUTPUT_DIR}/train.jsonl", train_clauses)
# save_jsonl(f"{OUTPUT_DIR}/val.jsonl", val_clauses)
# save_jsonl(f"{OUTPUT_DIR}/test.jsonl", test_clauses)

# Save splits for prev-1/prev/current format
save_jsonl(f"{OUTPUT_DIR}/train_prev.jsonl", train_clauses_prev)
save_jsonl(f"{OUTPUT_DIR}/val_prev.jsonl", val_clauses_prev)
save_jsonl(f"{OUTPUT_DIR}/test_prev.jsonl", test_clauses_prev)

print(f"\n✓ All splits saved to {OUTPUT_DIR}/")
print(f"\nFiles created:")
# print(f"  For prev/current/next format:")
# print(f"    - train.jsonl ({len(train_clauses)} clauses)")
# print(f"    - val.jsonl ({len(val_clauses)} clauses)")
# print(f"    - test.jsonl ({len(test_clauses)} clauses)")
print(f"  For prev-1/prev/current format:")
print(f"    - train_prev.jsonl ({len(train_clauses_prev)} clauses)")
print(f"    - val_prev.jsonl ({len(val_clauses_prev)} clauses)")
print(f"    - test_prev.jsonl ({len(test_clauses_prev)} clauses)")

### CLAUSE CLASSIFICATION TRAINING

### CONFIGURATIONS

In [ ]:
# MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
# MODEL_NAME = "law-ai/InCaseLawBERT"
# MODEL_NAME = "law-ai/InLegalBERT"
MODEL_NAME = "Saibo-creator/legal-roberta-base"


MAX_LEN = 512
OUTPUT_DIR = "./model_out_anti_overfit"
BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 2e-5
EARLY_STOPPING_PATIENCE = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABEL2ID = {"Premise": 0, "None": 1, "Opposition": 2, "Claim": 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Early Stopping Patience: {EARLY_STOPPING_PATIENCE} epochs")

### LOAD SPLIT DATA

In [ ]:
dataset = load_dataset("json", data_files={
    "train": "train_prev.jsonl",
    "validation": "val_prev.jsonl",
    "test": "test_prev.jsonl"
})

print("Dataset loaded:")
print(f"Train: {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")
print(f"Test: {len(dataset['test'])} samples")

### UPSAMPLE DATA


In [ ]:
def encode_labels(example):
    example["labels"] = LABEL2ID[example["label"]]
    return example

dataset = dataset.map(encode_labels)

print("\n📊 Analyzing class distribution before upsampling...")
train_data = dataset["train"]
train_labels = [example["labels"] for example in train_data]
label_counts = Counter(train_labels)
num_classes = len(LABEL2ID)

print("Original label distribution:")
for label_id in range(num_classes):
    label_name = ID2LABEL[label_id]
    count = label_counts.get(label_id, 0)
    print(f"  {label_name}: {count}")

max_count = max(label_counts.values()) if label_counts else 0
print(f"\nMaximum class count: {max_count}")

# Group training examples by label
train_by_label = defaultdict(list)
for example in train_data:
    label_id = example["labels"]
    train_by_label[label_id].append(example)

# Upsample minority classes
upsampled_train = []
for label_id in range(num_classes):
    label_name = ID2LABEL[label_id]
    examples = train_by_label[label_id]
    current_count = len(examples)

    if current_count == 0:
        print(f"⚠️ Warning: No examples found for {label_name}")
        continue

    if current_count < max_count:
        # Calculate how many times we need to repeat
        repeat_times = max_count // current_count
        remainder = max_count % current_count

        upsampled = examples * repeat_times

        if remainder > 0:
            random.shuffle(examples)
            upsampled.extend(examples[:remainder])

        upsampled_train.extend(upsampled)
        print(f"  {label_name}: {current_count} → {len(upsampled)} (upsampled)")
    else:
        upsampled_train.extend(examples)
        print(f"  {label_name}: {current_count} (no upsampling needed)")

# Shuffle the upsampled training data
random.shuffle(upsampled_train)

# Create new dataset with upsampled training data
upsampled_train_dataset = Dataset.from_list(upsampled_train)

# Update the dataset using DatasetDict
dataset = DatasetDict(
    {
        "train": upsampled_train_dataset,
        "validation": dataset["validation"],
        "test": dataset["test"],
    }
)

# Verify upsampling
final_label_counts = Counter([ex["labels"] for ex in upsampled_train_dataset])
print("\n✅ After upsampling:")
for label_id in range(num_classes):
    label_name = ID2LABEL[label_id]
    count = final_label_counts.get(label_id, 0)
    print(f"  {label_name}: {count}")

print(
    f"\nTotal training samples: {len(upsampled_train_dataset)} (was {len(train_data)})"
)


# Initialize Tokenizer
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


# Tokenization Function

def tokenize_fn(batch):
    """Tokenize text with context from prev_text and next_text"""
    texts = []
    for i in range(len(batch["text"])):
        text = batch["text"][i]
        prev = batch.get("prev_text", [None] * len(batch["text"]))[i]
        prev_1 = batch.get("prev_text_1", [None] * len(batch["text"]))[i]
        # next_text = batch.get("next_text", [None] * len(batch["text"]))[i]

        # Build context string
        context_parts = []
        if prev_1 and prev_1.strip():
            context_parts.append(f"[PREV_1] {prev_1.strip()}")
        if prev and prev.strip():
            context_parts.append(f"[PREV] {prev.strip()}")
        context_parts.append(f"[CURRENT] {text.strip()}")
        # if next_text and next_text.strip():
        #     context_parts.append(f"[NEXT] {next_text.strip()}")

        texts.append(" ".join(context_parts))

    return tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN)

print("Tokenizing datasets...")
tokenized_datasets = dataset.map(tokenize_fn, batched=True)

# Remove unnecessary columns
columns_to_remove = ["label", "doc_id", "clause_id"]
for col in columns_to_remove:
    if col in tokenized_datasets["train"].column_names:
        tokenized_datasets = tokenized_datasets.remove_columns([col])

# Set PyTorch format
tokenized_datasets.set_format(
    "torch", columns=["input_ids", "attention_mask", "labels"]
)

print("✅ Tokenization complete")

### INITIALIZE MODEL

In [ ]:
print(f"Loading model: {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)
model.to(DEVICE)
print("✅ Model loaded")

### COMPUTE METRICS

In [ ]:
WeightedTrainer = Trainer

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, target_names=LABEL2ID.keys(), output_dict=True, zero_division=0)

    metrics = {
        "f1_macro": report["macro avg"]["f1-score"],
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "accuracy": report["accuracy"]
    }

    # Add per-class F1 scores
    for label_name in LABEL2ID.keys():
        if label_name in report:
            metrics[f"{label_name}_f1"] = report[label_name]["f1-score"]

    return metrics

### TRAINING ARGUMENTS

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=2,

    # Learning rate and scheduling
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,

    # Regularization
    weight_decay=0.1,
    max_grad_norm=1.0,

    # Evaluation and saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    save_total_limit=3,

    # Reproducibility
    seed=SEED,
    data_seed=SEED,

    # Other settings
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
)

print("✅ Training arguments configured with anti-overfitting measures:")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Weight Decay: {training_args.weight_decay}")
print(f"  - Max Grad Norm: {training_args.max_grad_norm}")
print(f"  - LR Scheduler: {training_args.lr_scheduler_type}")
print(f"  - Early Stopping Patience: {EARLY_STOPPING_PATIENCE} epochs")

### INITIALIZE TRAINER WITH EARLY STOPPING

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

print("✅ Trainer initialized with early stopping callback")

### TRAIN MODEL

In [ ]:
print("🚀 Starting training...")
print("=" * 60)
trainer.train()
print("=" * 60)
print("✅ Training complete!")

### EVALUATE ON TEST SET

In [ ]:
print("\n📊 Evaluating on test set...")
test_metrics = trainer.evaluate(tokenized_datasets["test"])

print("\n" + "=" * 60)
print("TEST SET METRICS:")
print("=" * 60)
for key, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# Save test metrics
with open("test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=4)
print("\n✅ Test metrics saved to test_metrics.json")

### DETAILED CLASSIFICATION REPORT

In [ ]:
print("\n📈 Generating detailed classification report...")

# Get predictions on test set
predictions = trainer.predict(tokenized_datasets["test"])
y_pred = predictions.predictions.argmax(-1)
y_true = predictions.label_ids

report = classification_report(
    y_true, y_pred, target_names=list(LABEL2ID.keys()), output_dict=False
)

print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT:")
print("=" * 60)
print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

print("\n✅ Detailed report saved to classification_report.txt")

### SAVE FINAL MODEL

In [ ]:
final_model_path = "./final_model"
print(f"\n💾 Saving final model to {final_model_path}...")
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print("✅ Model and tokenizer saved!")